In [1]:
import numpy as np

In [2]:
n = 100_000
d = 3

mu = np.array([1, 10, 20]).reshape(-1, 1)
Sigma = np.array([1, -1, 0, -1, 3, -1, 0, -1, 3]).reshape(3, 3)

X = np.random.multivariate_normal(mean=mu.flatten(), cov=Sigma, size=n)

wt = 1  # Can modify this to take an exponentially weighted moving average
wtSum = 0.0
muhat = np.zeros((3, 1))

# Starting estimates of covariance / inverse covariance.
M = 0 * np.eye(d)
Minv = 100_000_000 * np.eye(d)
Chol = np.sqrt(100_000_000) * np.eye(d)

for x in X:
    x = x.reshape(-1, 1)
    wtSum = wt * wtSum + 1  # Divisor For Covariance Matrix
    delta = x - muhat
    muhat += delta / wtSum
    new_delta = (x - muhat).reshape(1, -1)

    # Estimation of Sigma
    M = wt * M + delta @ new_delta
    Sigma_hat = M / wtSum

    # Estimation of the inverse of Sigma
    Mnum = 1.0 / np.power(wt, 2) * Minv @ delta @ new_delta @ Minv
    Mden = 1.0 + 1.0 / wt * new_delta @ Minv @ delta
    Minv = Minv / wt - Mnum / Mden.item()

    Chol = Chol

print("Error between estimates (should be very small): ")
print(np.linalg.pinv(Sigma_hat) - Minv * wtSum)

print("Error between inverse of real and estimated covariance matrix:")
print(np.linalg.pinv(Sigma) - Minv * wtSum)

Error between estimates (should be very small): 
[[ 7.16982029e-13 -1.06115117e-12  8.38745740e-13]
 [ 1.27098332e-12 -1.06858966e-12  5.70127279e-13]
 [-7.64749375e-13  1.02587383e-12 -4.65960603e-13]]
Error between inverse of real and estimated covariance matrix:
[[0.0025169  0.00425884 0.00152781]
 [0.00425884 0.00275907 0.00158127]
 [0.00152781 0.00158127 0.00022925]]


# How to get the actual normalizer

That's great, but we don't just want to know the inverse covariance. When we use the inverse covariance, we use the _square root_ of it in normalization.

So what we really want is to know $M$, where $MM^\top = \Sigma^{-1}$.

Let's begin by writing out an update to an estimate of $\Sigma$, which we will call $S$.

$S + uv^\top$

This update is the online formula for covariance estimation, where $u = x_t - \hat{\mu}_{t-1}$ and $v = x_t - \hat{\mu}_t$. [See Wikipedia for details.](https://en.wikipedia.org/wiki/Algorithms_for_calculating_variance#Online), but it was the formulation I used above.

We want to know the inverse of this (and eventually the square root of this inverse):

$(S + uv^\top)^{-1} = S^{-1} - \frac{S^{-1} uv^\top S^{-1}}{1 + v^\top S^{-1} u}$

By the Sherman-Morrison formula. Suppose, however, that $S = LL^\top$, because it is positive definite, as a covariance matrix. Then we can write this as:

$(LL^\top)^{-1} - \frac{(LL^\top)^{-1} uv^\top (LL^\top)^{-1}}{1 + v^\top (LL^\top)^{-1} u}$

If we do a bit of algebra, we can pull out a term on the left and a corresponding one on the right:

$(L^{-1})^\top \left[ I - \frac{L^{-1} uv^\top (L^{-1})^\top}{1 + v^\top (L^{-1})^\top L^{-1} v}\right] L^{-1}$

Now call $x = L^{-1} u$ and $y = L^{-1} v$, just a rotated version of the original vectors, $u$ and $v$.

$(L^{-1})^\top \left[ I - \frac{xy^\top}{1 + y^\top x}\right] L^{-1}$.

Critically, the inner quantity, $I - \frac{xy^\top}{1 + y^\top x}$ admits a simple rank-one update to the cholesky of $I$.

So to update the Cholesky of the inverse ($L^{-1}$), we simply take this (easy to compute) Cholesky of the inner quantity. Apparently this is $O(d^2)$ time complexity using an algorithm of [Krause and Igel (2015)](https://dl.acm.org/doi/10.1145/2725494.2725496) (and [exposed in Tensorflow](https://www.tensorflow.org/probability/api_docs/python/tfp/math/cholesky_update), at least), which is cool, albeit kind of wasted, because we're still going to have to multiply $L^{-1}$ and the cholesky of the inner bit.

In [40]:
from scipy.linalg import cholesky

x = np.array([1, 2, 3, 4, 5]).reshape(-1, 1)
y = np.array([10, 11, 12, 13, 14]).reshape(-1, 1)

print(x, y)

print(x @ y.T)

U, D, V = np.linalg.svd(x @ y.T)
print(D)
U @ np.diag(D) @ V

print(U[:, 0], V[0, :], D[0])

U[:, [0]] @ V[[0], :] * D[0]

np.diff(U[:, 0]), np.diff(V[0, :])

[[1]
 [2]
 [3]
 [4]
 [5]] [[10]
 [11]
 [12]
 [13]
 [14]]
[[10 11 12 13 14]
 [20 22 24 26 28]
 [30 33 36 39 42]
 [40 44 48 52 56]
 [50 55 60 65 70]]
[2.00374649e+02 2.17283368e-14 8.28222771e-16 1.14707037e-30
 2.67094235e-31]
[-0.13483997 -0.26967994 -0.40451992 -0.53935989 -0.67419986] [-0.37011661 -0.40712827 -0.44413993 -0.48115159 -0.51816325] 200.37464909513872


(array([-0.13483997, -0.13483997, -0.13483997, -0.13483997]),
 array([-0.03701166, -0.03701166, -0.03701166, -0.03701166]))

In [48]:
L = cholesky(np.linalg.inv(np.eye(5) + x @ y.T), check_finite=True)

np.linalg.eig(L)

EigResult(eigenvalues=array([0.97347007, 0.9387847 , 0.88788084, 0.79613927, 0.52846029]), eigenvectors=array([[1.        , 0.86266813, 0.67286446, 0.4862471 , 0.27761473],
       [0.        , 0.50577039, 0.69400221, 0.68480433, 0.49184647],
       [0.        , 0.        , 0.256153  , 0.51146563, 0.58354759],
       [0.        , 0.        , 0.        , 0.1816857 , 0.51334386],
       [0.        , 0.        , 0.        , 0.        , 0.27743   ]]))

In [ ]:
n = 100_000
d = 3

mu = np.array([1, 10, 20]).reshape(-1, 1)
Sigma = np.array([1, -1, 0, -1, 3, -1, 0, -1, 3]).reshape(3, 3)

X = np.random.multivariate_normal(mean=mu.flatten(), cov=Sigma, size=n)

n = 0.0
muhat = np.zeros((3, 1))

# Starting estimates of covariance / inverse covariance.
M = 0 * np.eye(d)
Minv = 100_000_000 * np.eye(d)
Chol = np.sqrt(100_000_000) * np.eye(d)

for x in X:
    x = x.reshape(-1, 1)
    n += 1  # Divisor For Covariance Matrix
    delta = x - muhat
    muhat += delta / n
    new_delta = (x - muhat).reshape(1, -1)

    # Estimation of Sigma
    M = M + delta @ new_delta
    Sigma_hat = M / n

    # Estimation of the inverse of Sigma
    Mnum = Minv @ delta @ new_delta @ Minv
    Mden = 1.0 + new_delta @ Minv @ delta
    Minv = Minv - Mnum / Mden.item()

    Chol = Chol

print("Error between estimates (should be very small): ")
print(np.linalg.pinv(Sigma_hat) - Minv * wtSum)

print("Error between inverse of real and estimated covariance matrix:")
print(np.linalg.pinv(Sigma) - Minv * wtSum)

In [6]:
# def update_cholesky(L, x):
#     n = x.shape[0]
#     for k in range(n):
#         r = np.sqrt(np.power(L[k, k], 2) + np.power(x[k], 2));
#         c = r / L[k, k]
#         s = x[k] / L[k, k]
#         L[k, k] = r
#         if k < n:
#             L[(k+1):n, k] = (L[(k+1):n, k] + s * x[(k+1):n]) / c
#             x[(k+1):n] = c * x[(k+1):n] - s * L[(k+1):n, k]
#     return L


def update_cholesky(R, x):
    p = np.size(x)
    x = x.T
    for k in range(p):
        r = np.sqrt(np.power(R[k, k].item(), 2) + np.power(x[k].item(), 2))
        c = r / R[k, k].item()
        s = (x[k] / R[k, k]).item()
        R[k, k] = r
        R[k, k + 1 : p] = (R[k, k + 1 : p] + s * x[k + 1 : p]) / c
        x[k + 1 : p] = c * x[k + 1 : p] - s * R[k, k + 1 : p]
    return R


def downdate_cholesky(R, x):
    p = np.size(x)
    for k in range(p):
        r = np.sqrt(np.power(R[k, k], 2) - np.power(x[k], 2))
        c = r / R[k, k].item()
        s = (x[k] / R[k, k]).item()
        R[k, k] = r
        for i in range(k + 1, p):
            R[k, i] = (R[k, i] - s * x[i]) / c
            x[i] = c * x[i] - s * R[k, i]
    return R

In [ ]:
import numpy as np
from scipy.linalg import cholesky

N = 10
d = 3

mu = np.array([1, 10, 20]).reshape(-1, 1)
Sigma = np.array([1, -1, 0, -1, 3, -1, 0, -1, 3]).reshape(3, 3)

X = np.random.multivariate_normal(mean=mu.flatten(), cov=Sigma, size=N)

n = 0.0
muhat = np.zeros((3, 1))

# Starting estimates of covariance / inverse covariance.
M1 = 0 * np.eye(d)
M2 = 0 * np.eye(d)
Minv = 100_000_000 * np.eye(d)
Chol = 0 * np.eye(d)
Cholinv = np.eye(d)

for x in X:
    x = x.reshape(-1, 1)
    n += 1  # Divisor For Covariance Matrix
    delta = x - muhat
    muhat += delta / n
    new_delta = (x - muhat).reshape(1, -1)

    # Estimation of Sigma
    M1 = M1 + delta @ new_delta
    Sigma_hat1 = M1 / n

    M2 = M2 + (n - 1) / n * delta @ delta.T
    Sigma_hat2 = M2 / n

    print(n)
    print(Sigma_hat1)
    print(Sigma_hat2)

    # Estimation of the inverse of Sigma
    Mnum = Minv @ delta @ new_delta @ Minv
    Mden = 1.0 + new_delta @ Minv @ delta
    Minv = Minv - Mnum / Mden.item()

    if n > d:
        pass
        # print(n)
    frac = np.sqrt((n - 1) / n)
    try:
        Chol1 = update_cholesky(Chol, frac * delta.reshape(-1))
        v = Cholinv @ delta
        norm = frac / np.sqrt(1 + (v.T @ v).item())
        # print(norm)
        # Cholinv1 = Cholinv @ downdate_cholesky(np.eye(d), norm * v.reshape(-1))
        Cholinv1 = downdate_cholesky(Cholinv, (norm * Cholinv @ v).reshape(-1))
        if n > d:
            pass
            # print("r1")
            # print(Cholinv1)
    except Exception:
        pass
    try:
        Chol = cholesky(M2, check_finite=False)
        Cholinv = cholesky(Minv, check_finite=False)
        if n > d:
            pass
            # print("exp")
            # print(Cholinv)
    except Exception:
        pass

IndentationError: expected an indented block after 'if' statement on line 45 (3246729636.py, line 47)

In [25]:
Cholinv, Cholinv1

(array([[0.41434453, 0.10520804, 0.03153286],
        [0.        , 0.24687851, 0.07813035],
        [0.        , 0.        , 0.24007521]]),
 array([[0.42049381, 0.09091846, 0.00766703],
        [0.        , 0.25381219, 0.159132  ],
        [0.        , 0.        , 0.2111339 ]]))

In [150]:
N = 10
d = 3

mu = np.array([1, 10, 20]).reshape(-1, 1)
Sigma = np.array([1, -1, 0, -1, 3, -1, 0, -1, 3]).reshape(3, 3)

X = np.random.multivariate_normal(mean=mu.flatten(), cov=Sigma, size=N)

n = 0.0
muhat = np.zeros((3, 1))

# Starting estimates of covariance / inverse covariance.
M1 = 0 * np.eye(d)
M2 = 0 * np.eye(d)
Minv = 100_000_000 * np.eye(d)
Chol = 0 * np.eye(d)

for x in X:
    x = x.reshape(-1, 1)
    n += 1  # Divisor For Covariance Matrix
    delta = x - muhat
    muhat += delta / n
    new_delta = (x - muhat).reshape(1, -1)

    # Estimation of Sigma
    M1 = M1 + delta @ new_delta
    Sigma_hat1 = M1 / n

    M2 = M2 + (n - 1) / n * delta @ delta.T
    Sigma_hat2 = M2 / n

    # Estimation of the inverse of Sigma
    Mnum = Minv @ delta @ new_delta @ Minv
    Mden = 1.0 + new_delta @ Minv @ delta
    Minv = Minv - Mnum / Mden.item()

    # alpha = 1, beta = (n - 1) / n
    if n <= d:
        Chol = np.eye(d)
    elif n == d + 1:
        Chol = cholesky(Minv, check_finite=False, lower=False)
        print(Chol)
    else:
        if n % 1 == 0:
            print(n)
            print(Chol)
        # Chol = (
        #   Chol +
        #   1 / (delta.T @ delta) * (np.sqrt(1 + (n - 1) / n * (delta.T @ delta)) - 1) *
        #   (Chol @ delta) @ delta.T
        # )
        v = np.sqrt(1 - 1 / n) * Chol.T @ delta
        if np.allclose(v, np.zeros_like(v)):
            c = 0.5
        else:
            c = 1 / (v.T @ v) * (np.sqrt(1 + v.T @ v) - 1)
        # print(np.eye(d) + v @ v.T)
        sqrtv = np.eye(d) + c * v @ v.T
        # print(sqrtv @ sqrtv)
        print(c)
        # print(Chol / n)
        Chol = Chol @ (np.eye(d) + c * v @ v.T)

[[ 2.99150094  0.82131119  0.9034133 ]
 [ 0.          0.24206028 -0.10577202]
 [ 0.          0.          0.3506436 ]]
5.0
[[ 2.99150094  0.82131119  0.9034133 ]
 [ 0.          0.24206028 -0.10577202]
 [ 0.          0.          0.3506436 ]]
[[0.41071351]]
6.0
[[ 3.2472461   0.77891633  0.56234452]
 [ 0.01563942  0.23946773 -0.12662918]
 [-0.07245745  0.01201127  0.44727485]]
[[0.33236547]]
7.0
[[ 3.90556979  1.63416384 -0.47463556]
 [ 0.11832978  0.37287577 -0.28838525]
 [-0.22104592 -0.18102435  0.68132883]]
[[0.27293687]]
8.0
[[10.02958562  2.28873455  1.0796743 ]
 [ 0.24977339  0.38692523 -0.25502412]
 [-0.32539174 -0.19217744  0.65484527]]
[[0.13414827]]
9.0
[[63.16973516 17.31631709 -3.29969092]
 [ 2.15803657  0.92656584 -0.41228716]
 [-2.50214632 -0.80774513  0.83423512]]
[[0.00650329]]
10.0
[[9672.10300827 2581.81617545 -382.02918263]
 [ 344.7421121    92.35781926  -13.91500272]
 [-391.63445467 -104.66211911   16.17161724]]
[[0.00021708]]


In [147]:
print(cholesky(np.linalg.inv(Sigma)))

[[1.26491106 0.47434165 0.15811388]
 [0.         0.61237244 0.20412415]
 [0.         0.         0.57735027]]


In [95]:
print("Error between estimates (should be very small): ")
print(np.linalg.pinv(Sigma_hat1) - Minv * N)

print("Error between inverse of real and estimated covariance matrix:")
print(np.linalg.pinv(Sigma) - Minv * N)

print(Sigma_hat1 - Sigma_hat2)

# print(Chol @ Chol.T)
# print(Sigma_hat1)
print(Chol)
print(cholesky(np.linalg.inv(Sigma)))

Error between estimates (should be very small): 
[[ 4.08991951e-10  1.77469150e-11 -2.19571139e-10]
 [ 3.17339488e-11  3.14843929e-10  6.47946502e-10]
 [-1.66006514e-10  6.29431840e-10  1.52541141e-09]]
Error between inverse of real and estimated covariance matrix:
[[ 0.14470867  0.04566122  0.09467655]
 [ 0.04566122 -0.03232974  0.02708953]
 [ 0.09467655  0.02708953  0.0108552 ]]
[[ 0.00000000e+00 -1.11022302e-16  0.00000000e+00]
 [ 0.00000000e+00  8.88178420e-16 -1.11022302e-16]
 [ 8.32667268e-17  3.33066907e-16  8.88178420e-16]]
[[-2.78656122e+17  5.00096851e+17 -5.42361009e+17]
 [-1.09445782e+18  1.96419480e+18 -2.13019273e+18]
 [ 5.83967637e+18 -1.04803144e+19  1.13660262e+19]]
[[ 1.         -1.          0.        ]
 [ 0.          1.41421356 -0.70710678]
 [ 0.          0.          1.58113883]]
